# LangGraph + Follow Plan MCP 튜토리얼

이 튜토리얼에서는 **Follow Plan MCP 서버**를 LangGraph와 통합하여 프로젝트 계획 수립 및 작업 관리를 수행하는 방법을 학습합니다.

Follow Plan MCP 서버는 프로젝트의 작업(Task), 기능(Feature), 버그(Bug), 규칙(Rule)을 체계적으로 관리하고, SQLite 기반으로 영구 저장하며, 고급 검색 기능을 제공하는 도구를 제공합니다.

> 참고 문서: [Follow Plan MCP Server](https://github.com/vibeclasses/follow-plan-mcp)

## 학습 목표

- Follow Plan MCP 서버의 개념과 주요 도구를 이해합니다
- `create_agent`를 사용하여 Follow Plan 도구를 활용하는 에이전트를 생성합니다
- `ToolNode` 워크플로우에 Follow Plan을 통합합니다
- 프로젝트 계획 수립과 작업 추적에 활용하는 방법을 실습합니다

## 목차

1. 환경 설정
2. Follow Plan MCP 소개
3. MCP 서버 연결
4. `create_agent`를 활용한 Follow Plan
5. `ToolNode` 워크플로우와 Follow Plan 통합

## 환경 설정

튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로그 설정을 활성화하여 LangSmith에서 실행 과정을 확인할 수 있도록 합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)
# 추적을 위한 프로젝트 이름 설정
logging.langsmith("LangGraph-V1-Tutorial")

---

## Part 1: Follow Plan MCP 소개

### Follow Plan MCP란?

Follow Plan MCP 서버는 **프로젝트 계획 수립 및 작업 관리**를 지원하는 MCP 서버입니다. 프로젝트 디렉토리 내에 `.plan` 폴더를 생성하고, SQLite 데이터베이스(FTS5 전문 검색 지원)를 사용하여 작업, 기능, 버그, 규칙 등을 영구적으로 관리합니다.

### 주요 특징

- **작업 관리(Task)**: 프로젝트 작업을 생성, 수정, 조회, 삭제합니다
- **기능 관리(Feature)**: 기능 단위로 요구사항을 체계적으로 관리합니다
- **버그 추적(Bug)**: 버그를 기록하고 심각도별로 상태를 추적합니다
- **규칙 관리(Rule)**: 프로젝트 규칙과 가이드라인을 정의합니다
- **고급 검색(Search)**: FTS5 기반 전문 검색으로 모든 항목을 검색합니다
- **SQLite 영구 저장**: `.plan` 디렉토리에 데이터가 영구 보존됩니다

### 주요 도구 목록

| 카테고리 | 도구 | 설명 |
|---------|------|------|
| 작업 관리 | `create_task` | 새 작업을 생성합니다 |
| | `update_task` | 작업 정보를 수정합니다 |
| | `get_task` | 특정 작업을 조회합니다 |
| | `list_tasks` | 작업 목록을 조회합니다 |
| | `delete_task` | 작업을 삭제합니다 |
| 기능 관리 | `create_feature` | 새 기능을 생성합니다 |
| | `update_feature` | 기능 정보를 수정합니다 |
| | `list_features` | 기능 목록을 조회합니다 |
| 버그 추적 | `create_bug` | 새 버그를 등록합니다 |
| | `update_bug` | 버그 정보를 수정합니다 |
| | `list_bugs` | 버그 목록을 조회합니다 |
| 규칙 관리 | `create_rule` | 새 규칙을 생성합니다 |
| | `list_rules` | 규칙 목록을 조회합니다 |
| 검색 | `search` | 키워드로 전체 검색합니다 |
| | `advanced_search` | 고급 필터를 사용하여 검색합니다 |

### `create_task` 도구 주요 파라미터

| 파라미터 | 타입 | 필수 | 설명 |
|---------|------|------|------|
| `title` | string | ✅ | 작업 제목 |
| `description` | string | ❌ | 작업 상세 설명 |
| `status` | string | ❌ | 작업 상태 (todo, in_progress, done 등) |
| `priority` | string | ❌ | 우선순위 (low, medium, high, critical) |
| `feature_id` | string | ❌ | 연결된 기능 ID |

In [ ]:
import nest_asyncio
from typing import List, Dict, Any

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트: 여러 MCP 서버에 연결하여 도구를 가져옵니다
from langchain_mcp_adapters.client import MultiServerMCPClient

# 비동기 호출을 활성화합니다 (Jupyter 환경에서 필요)
nest_asyncio.apply()

In [ ]:
import sys, os, subprocess

# Windows + Jupyter workaround: MCP stdio passes Jupyter's sys.stderr to subprocess.Popen,
# but Jupyter's stderr doesn't support fileno(). Patch the default errlog to os.devnull.
if sys.platform == "win32":
    import mcp.client.stdio as _mcp_stdio

    _devnull_file = open(os.devnull, "w")

    # @asynccontextmanager wraps the original function — patch __wrapped__.__defaults__
    if hasattr(_mcp_stdio.stdio_client, "__wrapped__"):
        _mcp_stdio.stdio_client.__wrapped__.__defaults__ = (_devnull_file,)

    # Also patch the helper that creates the subprocess
    _mcp_stdio._create_platform_compatible_process.__defaults__ = (
        None,
        _devnull_file,
        None,
    )


async def setup_mcp_client(server_configs: dict):
    """MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 구성 딕셔너리. 각 서버의 이름을 키로,
                       연결 정보(command, args, transport 또는 url)를 값으로 가집니다.

    Returns:
        tuple: (MCP 클라이언트, 로드된 도구 목록)
    """
    # MCP 클라이언트 생성
    client = MultiServerMCPClient(server_configs)

    # 서버에 연결하여 도구 목록을 가져옵니다
    tools = await client.get_tools()

    # 로드된 도구 목록을 출력합니다
    print(f"[MCP] {len(tools)}개의 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")

    return client, tools

---

## Part 2: MCP 서버 연결

Follow Plan MCP 서버는 프로젝트 디렉토리 경로를 인자로 받아 실행됩니다. 해당 디렉토리에 `.plan` 폴더가 자동 생성되고, SQLite 데이터베이스에 모든 데이터가 저장됩니다.

> **참고**: `npx`를 사용하려면 Node.js 18 이상이 설치되어 있어야 합니다. Follow Plan MCP는 GitHub 리포지토리에서 직접 실행합니다.
>
> **대안**: `npx github:` 방식이 동작하지 않는 경우, 리포지토리를 클론하고 빌드한 후 `node dist/index.js <프로젝트경로>`로 실행할 수 있습니다.
> ```bash
> git clone https://github.com/vibeclasses/follow-plan-mcp.git
> cd follow-plan-mcp && npm install && npm run build
> ```

In [ ]:
# 프로젝트 작업 디렉토리 생성
PROJECT_DIR = os.path.abspath("./workspace/sample_project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"프로젝트 디렉토리: {PROJECT_DIR}")

# Follow Plan MCP 서버 구성 (stdio 전송 방식)
server_configs = {
    "follow_plan": {
        "command": "npx",
        "args": ["-y", "github:vibeclasses/follow-plan-mcp", PROJECT_DIR],
        "transport": "stdio",
    },
}

# MCP 클라이언트 생성 및 도구 로드
client, tools = await setup_mcp_client(server_configs=server_configs)

---

## Part 3: `create_agent`를 활용한 Follow Plan

에이전트를 생성하여 Follow Plan 도구를 활용합니다. 에이전트는 프로젝트 관리 요청을 받으면 자동으로 적절한 도구(`create_task`, `create_feature`, `list_tasks` 등)를 호출하여 작업을 수행합니다.

In [ ]:
# LLM 설정
# OpenAI 키 사용 시 gpt-5.2, gpt-4.1-mini 등으로 변경 가능
llm = init_chat_model("claude-sonnet-4-5", temperature=0)

# 에이전트 생성: Follow Plan 도구를 사용하는 에이전트
agent = create_agent(
    llm,
    tools,
    checkpointer=InMemorySaver(),  # 대화 상태를 메모리에 저장
)

In [ ]:
# 스트리밍 헬퍼 함수와 UUID 생성 함수를 import합니다
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

### 예제 1: 프로젝트 작업 생성

쇼핑몰 웹 프로젝트의 초기 작업 목록을 생성합니다. 에이전트가 `create_task` 도구를 활용하여 작업을 등록합니다.

In [ ]:
# 대화 스레드 ID를 설정합니다
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 프로젝트 작업 생성
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "쇼핑몰 웹 프로젝트를 시작합니다. 다음 3개의 작업을 생성해 주세요:\n1. 사용자 인증 시스템 구현 (우선순위: high)\n2. 상품 목록 페이지 개발 (우선순위: high)\n3. 장바구니 기능 구현 (우선순위: medium)")]},
    config=config,
)

### 예제 2: 기능 등록과 버그 추적

프로젝트 기능을 등록하고, 발견된 버그를 기록합니다. 에이전트가 `create_feature`와 `create_bug` 도구를 활용합니다.

In [ ]:
# 새로운 대화 스레드 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행: 기능 등록 및 버그 추적
response = await astream_graph(
    agent,
    inputs={"messages": [("human", "다음 작업을 수행해 주세요:\n1. '결제 시스템'이라는 기능(feature)을 생성해 주세요. 설명: PG사 연동을 통한 온라인 결제 처리 시스템\n2. '로그인 페이지 CSS 깨짐' 버그를 등록해 주세요. 설명: 모바일 환경에서 로그인 폼의 레이아웃이 깨지는 현상, 우선순위: high\n3. 현재 등록된 전체 작업 목록을 조회해 주세요.")]},
    config=config,
)

---

## Part 4: `ToolNode` 워크플로우와 Follow Plan 통합

보다 세밀한 제어가 필요한 경우 `ToolNode`와 `StateGraph`를 사용하여 커스텀 워크플로우를 구성할 수 있습니다. Tavily 검색 도구와 함께 사용하면 프로젝트 계획 관리와 실시간 정보 검색을 결합할 수 있습니다.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated, List, Dict, Any, TypedDict
from langchain_tavily import TavilySearch


class AgentState(TypedDict):
    """에이전트 상태 정의

    Attributes:
        messages: 대화 메시지 목록. add_messages 리듀서로 메시지가 누적됩니다.
        context: 추가 컨텍스트 정보를 저장하는 딕셔너리 (선택적)
    """

    messages: Annotated[List[BaseMessage], add_messages]
    context: Dict[str, Any]


async def create_mcp_workflow(server_configs: dict):
    """MCP 도구를 사용하는 커스텀 워크플로우를 생성합니다.

    이 함수는 MCP 도구와 Tavily 검색 도구를 결합하여
    에이전트-도구 루프를 구현하는 그래프를 생성합니다.

    Args:
        server_configs: MCP 서버 구성 딕셔너리

    Returns:
        CompiledStateGraph: 컴파일된 워크플로우 그래프
    """
    # MCP 클라이언트 생성 및 도구 로드
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # Tavily 웹 검색 도구 추가
    tavily_tool = TavilySearch(max_results=2)
    tools.append(tavily_tool)

    # LLM 설정 및 도구 바인딩
    # OpenAI 키 사용 시 gpt-5.2, gpt-4.1-mini 등으로 변경 가능
    llm = init_chat_model("claude-sonnet-4-5", temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    # 워크플로우 그래프 생성
    workflow = StateGraph(AgentState)

    async def agent_node(state: AgentState):
        """에이전트 노드: LLM을 호출하여 응답을 생성합니다"""
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    # ToolNode 생성: 도구 호출을 처리합니다
    tool_node = ToolNode(tools)

    # 그래프에 노드 추가
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)

    # 엣지 정의: 시작 -> 에이전트
    workflow.add_edge(START, "agent")

    # 조건부 엣지: 에이전트 -> (도구 or 종료)
    # tools_condition은 도구 호출이 필요하면 "tools"로, 아니면 END로 라우팅합니다
    workflow.add_conditional_edges("agent", tools_condition)

    # 도구 -> 에이전트 (도구 실행 후 다시 에이전트로)
    workflow.add_edge("tools", "agent")

    # 그래프 컴파일
    app = workflow.compile(checkpointer=InMemorySaver())

    return app

In [ ]:
# Follow Plan + Tavily 워크플로우 생성
server_configs = {
    "follow_plan": {
        "command": "npx",
        "args": ["-y", "github:vibeclasses/follow-plan-mcp", PROJECT_DIR],
        "transport": "stdio",
    },
}

mcp_app = await create_mcp_workflow(server_configs)

In [ ]:
from IPython.display import Image

# 컴파일된 워크플로우 그래프를 시각화합니다
Image(mcp_app.get_graph().draw_mermaid_png())

### 예제 3: 프로젝트 계획 + 웹 검색 결합

Follow Plan으로 프로젝트를 체계적으로 관리하면서, Tavily 검색으로 최신 기술 정보를 수집하여 작업에 반영하는 복합 작업을 수행합니다.

In [ ]:
# 새로운 대화 스레드 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 워크플로우 실행: 프로젝트 계획 + 웹 검색
_ = await astream_graph(
    mcp_app,
    inputs={
        "messages": [
            (
                "human",
                "React와 Next.js를 사용한 쇼핑몰 프론트엔드 프로젝트를 계획하고 있습니다. "
                "최신 Next.js 버전의 주요 기능을 웹에서 검색한 후, "
                "검색 결과를 바탕으로 '프론트엔드 아키텍처 설계'라는 기능(feature)을 생성하고, "
                "관련 작업 3개를 생성해 주세요.",
            )
        ]
    },
    config=config,
)

---

## 마무리

이 튜토리얼에서는 Follow Plan MCP 서버를 LangGraph와 통합하는 방법을 학습했습니다.

### 핵심 정리

- **Follow Plan MCP 서버**는 `npx github:vibeclasses/follow-plan-mcp <프로젝트경로>`로 프로젝트 디렉토리를 지정하여 실행할 수 있습니다
- 프로젝트 디렉토리 내 `.plan` 폴더에 SQLite 데이터베이스가 생성되어 데이터가 영구 저장됩니다
- `create_task`, `create_feature`, `create_bug`, `create_rule` 등의 도구로 프로젝트를 체계적으로 관리합니다
- `search`, `advanced_search` 도구로 등록된 항목을 효율적으로 검색할 수 있습니다
- `create_agent`로 간단한 에이전트를, `ToolNode` + `StateGraph`로 커스텀 워크플로우를 구성할 수 있습니다
- Tavily 등 다른 도구와 결합하면 실시간 정보를 반영한 프로젝트 계획이 가능합니다

### 활용 팁

- AI 에이전트가 프로젝트 계획을 자동으로 수립하고 관리하도록 활용하세요
- 버그 추적과 기능 관리를 결합하면 스프린트 단위의 프로젝트 관리가 가능합니다
- 규칙(Rule)을 등록하여 프로젝트 코딩 가이드라인이나 아키텍처 원칙을 관리할 수 있습니다